In [ ]:
library(Signac)
library(Seurat)
library(ggplot2)
library(future)

plan("multicore", workers = 12)
options(future.globals.maxSize = 100000 * 1024^3)

set.seed(1234)

The legacy packages maptools, rgdal, and rgeos, underpinning this package
will retire shortly. Please refer to R-spatial evolution reports on
https://r-spatial.org/r/2023/05/15/evolution4.html for details.
This package is now running under evolution status 0 

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, saveRDS


Loading Seurat v5 beta version 
To maintain compatibility with previous workflows, new Seurat objects will use the previous object structure by default
To use new Seurat v5 assays: Please run: options(Seurat.object.assay.version = 'v5')



In [ ]:
atac <- readRDS('data/atac_common_peak_set_dmg_atlas_qc_filtered_dbl_scores_merged.rds')
multiome <- readRDS('data/multiome_common_peak_set_dmg_atlas_qc_filtered_dbl_scores_merged.rds')

In [ ]:
# first add dataset-identifying metadata
atac$dataset <- "ATAC"
multiome$dataset <- "Multiome"

# merge
dmg <- merge(atac, multiome)

# process the combined dataset
dmg <- FindTopFeatures(dmg, min.cutoff = 10)
dmg <- RunTFIDF(dmg)
dmg <- RunSVD(dmg)
dmg <- RunUMAP(dmg, reduction = "lsi", dims = 2:30)

Performing TF-IDF normalization

Running SVD

Scaling cell embeddings

Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”
08:25:13 UMAP embedding parameters a = 0.9922 b = 1.112

Found more than one class "dist" in cache; using the first, from namespace 'BiocGenerics'

Also defined by ‘spam’

08:25:13 Read 143344 rows and found 29 numeric columns

08:25:13 Using Annoy for neighbor search, n_neighbors = 30

Found more than one class "dist" in cache; using the first, from namespace 'BiocGenerics'

Also defined by ‘spam’

08:25:13 Building Annoy index with metric = cosine, n_trees = 50

0%   10   20   30   40   50   60   70   80   90   100%

[----|----|----|----|----|----|----|----|----|----|

*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*
*

In [ ]:
dmg

An object of class Seurat 
378298 features across 143344 samples within 1 assay 
Active assay: ATAC (378298 features, 378298 variable features)
 2 layers present: counts, data
 2 dimensional reductions calculated: lsi, umap

### Integration

In [ ]:
# compute LSI
atac <- FindTopFeatures(atac, min.cutoff = 10)
atac <- RunTFIDF(atac)
atac <- RunSVD(atac)

Performing TF-IDF normalization

Warning message in RunTFIDF.default(object = GetAssayData(object = object, slot = "counts"), :
“Some features contain 0 total counts”
Running SVD

Scaling cell embeddings



In [ ]:
# compute LSI
multiome <- FindTopFeatures(multiome, min.cutoff = 10)
multiome <- RunTFIDF(multiome)
multiome <- RunSVD(multiome)

Performing TF-IDF normalization

Running SVD

Scaling cell embeddings



In [ ]:
# find integration anchors
integration.anchors <- FindIntegrationAnchors(
  object.list = list(multiome, atac),
  anchor.features = rownames(multiome),
  reduction = "rlsi",
  dims = 2:30
)

# integrate LSI embeddings
integrated <- IntegrateEmbeddings(
  anchorset = integration.anchors,
  reductions = dmg[["lsi"]],
  new.reduction.name = "integrated_lsi",
  dims.to.integrate = 1:30
)

# create a new UMAP using the integrated embeddings
integrated <- RunUMAP(integrated, reduction = "integrated_lsi", dims = 2:30)

Computing within dataset neighborhoods

Finding all pairwise anchors

Warning message:
“`invoke()` is deprecated as of rlang 0.4.0.
Please use `exec()` or `inject()` instead.
This warning is displayed once every 8 hours.”
Converting layer counts in assay ATAC to empty dgCMatrix

Converting layer counts in assay ATAC to empty dgCMatrix

Warning message:
“No filtering performed if passing to data rather than counts”
Projecting new data onto SVD

Projecting new data onto SVD

Finding neighborhoods

Finding anchors

	Found 5028 anchors

Warning message:
“`invoke()` is deprecated as of rlang 0.4.0.
Please use `exec()` or `inject()` instead.
This warning is displayed once every 8 hours.”
Merging dataset 2 into 1

Extracting anchors for merged samples

Finding integration vectors

Finding integration vector weights

Integrating data

08:59:52 UMAP embedding parameters a = 0.9922 b = 1.112

Found more than one class "dist" in cache; using the first, from namespace 'BiocGenerics'

Also defined 

In [ ]:
rm(atac)
rm(multiome)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,8418660,449.7,14211632,759.0,14211632,759.0
Vcells,9120415066,69583.3,20894775873,159414.5,16898630968,128926.4


### Create a gene activity matrix

In [ ]:
gene.activities <- GeneActivity(integrated)

Extracting gene coordinates

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extracting reads overlapping genomic regions

Extra

In [ ]:
# add the gene activity matrix to the Seurat object as a new assay and normalize it
integrated[['RNA']] <- CreateAssayObject(counts = gene.activities)
integrated <- NormalizeData(
  object = integrated,
  assay = 'RNA',
  normalization.method = 'LogNormalize',
  scale.factor = median(integrated$nCount_RNA)
)

### Integrating with scRNA-seq data

In [ ]:
dmg_rna <- readRDS("/projects/0/einf2548/cruiz/dmg/data/merged_dmg_atlas_qc_filtered.rds")

dmg_rna <- AddMetaData(dmg_rna, readRDS('/projects/0/einf2548/cruiz/dmg/data/iCNV_dmg_atlas.rds'))

dmg_rna$cell_id <- paste0(dmg_rna$predicted.annotation_level_3, '_', dmg_rna$iCNV)

In [ ]:
dmg_rna$cell_id <- dplyr::recode(dmg_rna$cell_id,
                  `NPC-like_normal` = 'Stem_like_normal', 
                   `OPC_normal` = 'Stem_like_normal', 
                   `OPC-like_normal` = 'Stem_like_normal', 
                   `Endothelial_normal` = 'Endothelial', 
                   `TAM-BDM_normal` = 'TAM_BDM', 
                   `Mono_normal` = 'Mono', 
                   `Oligodendrocyte_normal` = 'Oligodendrocyte', 
                   `CD4/CD8_normal` = 'CD4_CD8', 
                   `MES-like_normal` = 'Differentiated_like_normal', 
                   `TAM-MG_normal` = 'TAM_MG', 
                   `B cell_normal` = 'B_cell', 
                   `DC_normal` = 'DC', 
                   `Mural cell_normal` = "Mural_cell", 
                   `Endothelial_tumor` = 'Endothelial', 
                   `TAM-BDM_tumor` = 'TAM_BDM', 
                   `Mono_tumor` = 'Mono', 
                   `Plasma B_normal` = 'Plasma', 
                   `Oligodendrocyte_tumor` = 'Oligodendrocyte', 
                   `Mural cell_tumor` = 'Mural_cell', 
                   `Plasma B_tumor` = 'Plasma', 
                   `MES-like_tumor`  = 'Differentiated_like_tumor', 
                   `CD4/CD8_tumor` = 'CD4_CD8', 
                   `OPC-like_tumor` = 'Stem_like_tumor', 
                   `AC-like_tumor` = 'Differentiated_like_tumor', 
                   `NPC-like_tumor` = 'Stem_like_tumor', 
                   `AC-like_normal` = 'Differentiated_like_normal', 
                   `TAM-MG_tumor` = 'TAM_MG', 
                   `B cell_tumor` = 'B_cell', 
                   `OPC_tumor` = 'Stem_like_tumor', 
                   `Neuron_normal` = 'Neuron', 
                   `Mast_tumor` = 'Mast', 
                   `Astrocyte_tumor` = 'Differentiated_like_tumor', 
                   `Neuron_tumor` = 'Neuron', 
                   `Astrocyte_normal` = 'Differentiated_like_normal', 
                   `DC_tumor` = 'DC', 
                   `Mast_normal` = 'Mast',
                   `NK_normal` = 'CD4_CD8')

In [ ]:
n_cells <- nrow(dmg_rna@meta.data)
n_sample <- round(n_cells * 0.3)
sampled_cells <- sample(Cells(dmg_rna), n_sample)
subset_dmg <- subset(dmg_rna, cells = sampled_cells)
subset_dmg

An object of class Seurat 
38576 features across 122868 samples within 6 assays 
Active assay: RNA (19248 features, 2000 variable features)
 3 layers present: counts, data, scale.data
 5 other assays present: RAW, prediction.score.annotation_level_1, prediction.score.annotation_level_2, prediction.score.annotation_level_3, prediction.score.annotation_level_4
 4 dimensional reductions calculated: pca, umap, ref.pca, ref.umap

In [ ]:
rm(dmg_rna)
gc()

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,11145047,595.3,20395044,1089.3,20395044,1089.3
Vcells,19918624051,151967.1,30088653256,229558.3,20888360267,159365.6


In [ ]:
transfer.anchors <- FindTransferAnchors(
  reference = subset_dmg,
  query = integrated,
  reduction = 'cca'
)

Running CCA



In [ ]:
predicted.labels.broad <- TransferData(
  anchorset = transfer.anchors,
  refdata = subset_dmg$cell_id,
  weight.reduction = integrated[['integrated_lsi']],
  dims = 2:30
)

predicted.labels.broad

In [ ]:
predicted.labels.fine <- TransferData(
  anchorset = transfer.anchors,
  refdata = subset_dmg$predicted.annotation_level_3,
  weight.reduction = integrated[['integrated_lsi']],
  dims = 2:30
)

predicted.labels.fine

In [ ]:
integrated@misc$predicted.labels.fine <- predicted.labels.fine

In [ ]:
integrated <- AddMetaData(object = integrated, metadata = predicted.labels.broad)

In [ ]:
plot1 <- DimPlot(
  object = subset_dmg,
    cols = dittoSeq::dittoColors(),
  group.by = 'cell_id',
  # label = TRUE,
  repel = TRUE) + NoAxes() + ggtitle('scRNA-seq')

plot2 <- DimPlot(
  object = integrated,
    cols = dittoSeq::dittoColors(),
  group.by = 'predicted.id',
  # label = TRUE,
  repel = TRUE) + NoAxes() + ggtitle('scATAC-seq')

options(repr.plot.height = 6, repr.plot.width = 14)
plot1 + plot2

In [ ]:
saveRDS(integrated, 'data/cca_integrated_atac_multiome_dmg_atlas.rds')